# Silver Layer — Patient ICD Diagnoses (`diagnoses_icd`)

**Source -> Target:** `<source_schema>.diagnoses_icd` -> `<target_schema>.diagnoses_icd` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Objective:** Standardize patient-level ICD diagnosis records while preserving patient identifiers, hospital admission relationships, diagnosis sequence, and ICD-9/ICD-10 distinctions.

**Expected grain:** One diagnosis assignment per patient, hospital admission and diagnosis sequence (`subject_id`, `hadm_id`, `seq_num`; all three NOT NULL per MIMIC-IV `hosp.diagnoses_icd`). Uniqueness of the full diagnosis key (`subject_id`, `hadm_id`, `seq_num`, `icd_code`, `icd_version`) is asserted; uniqueness of the 3-column documented key is not separately asserted (avoided to limit full-table shuffles on this large table; confirm during profiling). `icd_code`/`icd_version` link to `d_icd_diagnoses`; no join is performed here (reference coverage is a Gold/profiling concern).

**Source-file evidence (local MIMIC-IV CSV, 6,364,488 rows):** the documented 3-column key `(subject_id, hadm_id, seq_num)` is **not unique** in the data (6,364,372 distinct keys, i.e. 116 rows share a key with another row), whereas the full 5-column record is unique (6,364,488 distinct) and no column contains NULLs. The unconditional uniqueness check is therefore on the 5-column record, not the 3-column key. See `docs/gold_governance_checks.md`, Source readiness.

**Scale:** this is a large table (multi-million rows), so validation uses single-pass aggregates and no `exceptAll` comparisons.

**Silver rules:** Preserve clinical meaning and source records; standardize types and whitespace; validate key completeness, uniqueness and ICD versions; avoid imputation, unjustified deletion and analytical aggregation.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "diagnoses_icd"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "seq_num": "int",
    "icd_code": "string",
    "icd_version": "int",
}

In [0]:
# Load and inspect the source diagnosis records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))
bronze_count = bronze_df.count()
print("Bronze rows:", bronze_count)

In [0]:
# Profile record counts and missing values in source columns
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"]
    ]
)

display(bronze_profile)

In [0]:
# Standardize data types and ICD code whitespace without dropping records
silver_df = bronze_df.select(
    F.col("subject_id").cast("long").alias("subject_id"),
    F.col("hadm_id").cast("long").alias("hadm_id"),
    F.col("seq_num").cast("int").alias("seq_num"),
    F.trim(F.col("icd_code").cast("string")).alias("icd_code"),
    F.col("icd_version").cast("int").alias("icd_version")
)

display(silver_df.limit(20))
silver_df.printSchema()

In [0]:
# Check duplicate patient diagnosis assignments using the full diagnosis key
duplicate_df = (
    silver_df
    .groupBy(
        "subject_id",
        "hadm_id",
        "seq_num",
        "icd_code",
        "icd_version"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print("Duplicate full diagnosis records:", duplicate_count)
display(duplicate_df)

In [0]:
# Check missing identifiers, blank ICD codes and invalid ICD versions
quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"]
    ],
    F.sum(
        F.when(F.col("icd_code") == "", 1).otherwise(0)
    ).alias("blank_icd_codes"),
    F.sum(
        F.when(
            F.col("icd_version").isNotNull() &
            (~F.col("icd_version").isin(9, 10)), 1
        ).otherwise(0)
    ).alias("invalid_icd_versions")
)

display(quality_df)

In [0]:
# Confirm that Silver transformation preserves all source records
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

In [0]:
# Validate diagnosis uniqueness, data quality and record preservation
quality = quality_df.first().asDict()

assert duplicate_count == 0, f"Duplicate diagnosis records: {duplicate_count}"
assert all(value == 0 for value in quality.values()), f"Quality issues: {quality}"
assert bronze_count == silver_count, "Bronze and Silver counts do not match"

print("All critical Silver quality checks passed.")

In [0]:
# Publish validated diagnosis records (CREATE OR REPLACE; downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

In [0]:
# Verify the published Silver table and its record count
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

# Key checks on the published table in one pass
published_quality = published_df.select(
    F.sum(
        (F.col("subject_id").isNull() | F.col("hadm_id").isNull() | F.col("seq_num").isNull()).cast("int")
    ).alias("null_keys"),
    F.sum(F.col("icd_code").isNull().cast("int")).alias("null_icd_code"),
    F.sum(F.col("icd_version").isNull().cast("int")).alias("null_icd_version")
).first()
assert all(v == 0 for v in published_quality.asDict().values()), f"Published NULLs: {published_quality}"

published_duplicates = (
    published_df
    .groupBy("subject_id", "hadm_id", "seq_num", "icd_code", "icd_version")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert published_duplicates == 0, f"Published duplicate diagnosis keys: {published_duplicates}"

display(published_df.limit(20))

print("Silver diagnoses_icd table verified successfully.")

In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")